[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/intronlp-2026/blob/main/notebooks/week-04.ipynb)

# 4주차 실습: 형태소 분석기로 품사 붙이기

**목표.** 형태소 분석 라이브러리 eKoNLPy로 문장을 조각내고 품사를 붙인 뒤, 사람이 붙인 것과 다른 자리를 찾는다.

## 0. 준비

형태소 분석 라이브러리 eKoNLPy를 설치합니다. 사전 파일을 함께 내려받기 때문에 처음에는 시간이 조금 걸립니다. 한 번만 하면 됩니다.

설치 중에 이미 깔려 있던 다른 라이브러리와 버전이 맞지 않는다는 붉은 경고(`dependency conflicts`)가 나올 수 있습니다. **경고일 뿐이니 그대로 다음 셀로 넘어가면 됩니다.** 1-1 셀이 오류 없이 돌면 설치는 성공입니다.

In [ ]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install ekonlpy

## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.

### 1-1. 분석기 불러오기

문장을 넣으면 `(조각, 태그)` 짝의 목록이 나옵니다.

In [ ]:
from ekonlpy.tag import Mecab

tagger = Mecab()
print(tagger.pos("나는 성산으로 갔다"))

### 1-2. 태그 대응표

분석기의 태그는 `NNG`, `JKB` 같은 낯선 약자이고 수업에서 쓴 여섯 가지보다 훨씬 잘게 나뉘어 있습니다. 외울 필요는 없습니다. 아래 `show` 가 약자 옆에 풀이와 수업 태그를 붙여 줍니다.

| 분석기 태그 | 수업 태그 |
|---|---|
| NNG, NNP, NNB, NP, NR 등 N으로 시작 | 명사 |
| J로 시작 (JKS, JKO, JKB, JX 등) | 조사 |
| VV, VX, XSV (물질**하**는의 하) | 동사 |
| VA, XSA (신기**했**다의 했) | 형용사 |
| MAG, MAJ | 부사 |
| E로 시작 (EP, EC, EF, ETM 등) | 어미 |
| 그 밖 (XSV, MM, 기호 등) | 기타 |

In [ ]:
import pandas as pd

def course_tag(tag):
    tag = tag.split("+")[0]
    if tag.startswith("N"):
        return "명사"
    if tag.startswith("J"):
        return "조사"
    if tag in ("VV", "VX", "XSV"):
        return "동사"
    if tag in ("VA", "XSA"):
        return "형용사"
    if tag in ("MAG", "MAJ"):
        return "부사"
    if tag.startswith("E"):
        return "어미"
    return "기타"

def show(sentence):
    rows = [(piece, tag, tagger.tagset.get(tag.split("+")[0], ""), course_tag(tag))
            for piece, tag in tagger.pos(sentence)]
    print(sentence, "->", len(rows), "조각")
    return pd.DataFrame(rows, columns=["조각", "태그", "풀이", "수업 태그"])

show("나는 성산으로 갔다")

### 1-3. 지정 문장 1: 사람이 붙인 것과 나란히

사람은 `새별오름에서 쇠소깍까지 갔다` 를 여섯 조각으로 봅니다. 분석기 출력과 조각 경계부터 맞는지 봅니다. **조각 경계가 어긋나면 그 뒤의 이름표는 비교할 수조차 없습니다.**

In [ ]:
sentence = "새별오름에서 쇠소깍까지 갔다"
human = ["새별오름/명사", "에서/조사", "쇠소깍/명사", "까지/조사", "가/동사", "았다/어미"]
analyzer = [f"{p}/{course_tag(t)}" for p, t in tagger.pos(sentence)]

print("사람  ", len(human), "조각:", " · ".join(human))
print("분석기", len(analyzer), "조각:", " · ".join(analyzer))

### 1-4. 지정 문장 2, 3

- `성산일출봉 앞에서 물질하는 걸 보난 신기했다`: 성산일출봉이 통째로 잡히는지, 보난(보니까)이 어떻게 잘리는지, 걸이 "것"과 "을"로 나뉘는지 봅니다
- `혼저 옵서예`: 사전에 없는 방언이 어떻게 처리되는지 봅니다

In [ ]:
show("성산일출봉 앞에서 물질하는 걸 보난 신기했다")

In [ ]:
show("혼저 옵서예")

다른 자리를 찾으면 오류 유형 코드를 붙입니다.

| 코드 | 유형 |
|---|---|
| 가 | 고유명사 쪼개짐 |
| 나 | 고유명사를 일반명사로 |
| 다 | 사전에 없는 말을 아는 조각으로 억지로 맞춤 |
| 라 | 조사 잘못 나눔 |
| 마 | 품사 헷갈림 |

## 2. 한 지점만 바꿔 보기

아래 셀의 `TODO` 로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 바꾸기 전 결과를 먼저 확인해 두면 무엇이 달라졌는지 비교할 수 있습니다.

따옴표 안에 내 문장을 넣습니다. 온라인 활동 A의 `하늘을 나는 새가 아주 빠르다`, `비가 와서 한라산에 못 갔다` 나 방언 문장 `폭삭 속았수다` 를 넣어 보세요.

In [ ]:
# TODO: 따옴표 안의 문장을 바꿔 보세요
sentence = "나는 성산으로 갔다"

# 아래는 그대로 둡니다
show(sentence)

## 3. 확인 질문

1. 지정 문장 세 개의 출력을 옮기고, 사람이 붙인 것과 다른 자리에 표시하세요.
2. 다른 자리마다 오류 유형 코드(가~마)를 붙이세요.
3. 흔한 말과 드문 말(지명, 방언) 중 어느 쪽에서 더 많이 어긋났나요? 왜 그렇다고 생각하나요?

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.

*(여기에 답을 적으세요)*

## 4. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/intronlp-2026)의 `assignments/week-04/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.